# 04 — Control del sesgo de supervivencia (IEC15)

**Problema:** el universo del notebook 03 solo tiene las acciones que cotizan **hoy**. Si una empresa estuvo entre las más líquidas en 2019–2022 y después se deslistó o cambió de ticker, el backtest la omitiría y quedaría sesgado.

**Qué se prueba aquí:**
1. **Posibles deslistadas** que estuvieron en el IPSA desde 2019: ¿Yahoo todavía tiene sus datos?
2. **Empresas que cambiaron de nombre o ticker:** ¿Yahoo guarda su historia completa bajo el ticker actual o bajo el antiguo?

Los tickers antiguos son **hipótesis a probar**: si Yahoo no entrega datos, eso también es un resultado y se documenta.

Fuentes: DF (22-mar-2022) informó que el S&P IPSA pasó de 30 a 28 acciones tras la salida de ILC y AES Andes por deslistamiento. DF (28-ago-2025) mencionó la fusión de Security con Bicecorp.

Antes de hacer commit: **Edit → Clear Outputs of All Cells**.

In [ ]:
from pathlib import Path
import pandas as pd
import yfinance as yf

INICIO, FIN = "2019-01-01", "2026-07-16"   # FIN es exclusivo: llega al 15-jul-2026

PRUEBAS = {
    # Posibles deslistadas o fusionadas
    "AESANDES.SN":   "AES Andes (ex AES Gener) — salió del IPSA por deslistamiento",
    "AESGENER.SN":   "AES Gener — ticker anterior de AES Andes",
    "SECURITY.SN":   "Grupo Security — fusionado con Bicecorp",
    "NUEVAPOLAR.SN": "La Polar — salió del IPSA en 2018",
    # Tickers antiguos de empresas que cambiaron de nombre
    "ITAUCORP.SN":   "Itaú Corpbanca — ticker anterior de ITAUCL",
    "CENCOSHOPP.SN": "Cencosud Shopping — ticker anterior de CENCOMALLS",
    "OROBLANCO.SN":  "Oro Blanco — ticker anterior de PAMPA",
    # Tickers actuales: ¿su historia parte en 2019?
    "ITAUCL.SN":     "Banco Itaú Chile (actual)",
    "CENCOMALLS.SN": "Cencosud Shopping (actual)",
    "PAMPA.SN":      "Pampa Investments (actual)",
    "ILC.SN":        "ILC — sigue cotizando, salió del IPSA",
}

## Descargar y resumir

Para cada ticker: cuántos días tiene datos, desde y hasta cuándo, y la **mediana del valor transado diario** en el segundo semestre de 2019 (la ventana de la primera selección del índice). Con eso se ve si la empresa podría haber estado entre las 15 más líquidas.

In [ ]:
filas = []
for t, desc in PRUEBAS.items():
    try:
        df = yf.Ticker(t).history(start=INICIO, end=FIN, auto_adjust=False)
    except Exception as e:
        df = pd.DataFrame()
    if df.empty:
        filas.append({"ticker": t, "descripcion": desc, "resultado": "SIN DATOS"})
        continue
    df.index = df.index.tz_localize(None)
    valor = (df["Close"] * df["Volume"]) / 1e6                      # millones de CLP
    v2019 = valor.loc["2019-06-01":"2019-11-15"]
    filas.append({
        "ticker": t, "descripcion": desc, "resultado": "OK",
        "dias": len(df),
        "primera_fecha": df.index.min().date(),
        "ultima_fecha": df.index.max().date(),
        "mediana_valor_2S2019_mm": round(v2019.median(), 1) if len(v2019) else None,
    })

resumen = pd.DataFrame(filas)
resumen

## Cómo leer el resultado

- **SIN DATOS** en un ticker antiguo → Yahoo no guarda esa historia con ese nombre.
- Un ticker **actual** con `primera_fecha` en 2019 → Yahoo guarda la historia completa bajo el nombre nuevo, y no hay que hacer nada.
- Una empresa deslistada **con datos** → se agrega al universo hasta su última fecha.
- Una empresa deslistada **sin datos** y con liquidez alta en 2019 → es un hueco que se declara como limitación del backtest.

**Envíame una captura de la tabla.**